In [0]:
CREATE SCHEMA IF NOT EXISTS retail_q.semantic_layer;

CREATE OR REPLACE VIEW retail_q.semantic_layer.retail_metrics
WITH METRICS LANGUAGE YAML AS
$$
version: 1.1
comment: "Retail sales KPIs over the gold star schema (fact_sales + calendar, customer, product dimensions)"
source: retail_q.gold.fact_sales

joins:
- name: calendar
  source: retail_q.gold.dim_calendar
  'on': source.date_key = calendar.date_key
  rely:
    at_most_one_match: true

- name: customer
  source: retail_q.gold.dim_customer
  'on': source.customer_id = customer.customer_id
  rely:
    at_most_one_match: true

- name: product
  source: retail_q.gold.dim_product
  'on': source.product_id = product.product_id
  rely:
    at_most_one_match: true

fields:
# --- Time (from dim_calendar) ---
- name: Transaction Date
  expr: calendar.full_date
  comment: "Date of the sale"

- name: Year
  expr: calendar.year
  comment: "Calendar year of the sale"

- name: Quarter
  expr: calendar.quarter
  comment: "Calendar quarter of the sale (1-4)"

- name: Month
  expr: calendar.month_name
  comment: "Calendar month name of the sale"

- name: Day of Week
  expr: calendar.day_name
  comment: "Day name of the sale"

- name: Is Weekend
  expr: calendar.is_weekend
  comment: "Whether the sale happened on a weekend"

# --- Customer (from dim_customer) ---
- name: Customer Type
  expr: customer.customer_type
  comment: "Type of the customer"

- name: City
  expr: customer.city
  comment: "Customer city"

- name: State
  expr: customer.state
  comment: "Customer state"

- name: Country
  expr: customer.country
  comment: "Customer country"

- name: Industry
  expr: customer.industry
  comment: "Customer industry"

# --- Product (from dim_product) ---
- name: Category
  expr: product.category
  comment: "Product category"

- name: Subcategory
  expr: product.subcategory
  comment: "Product subcategory"

- name: Brand
  expr: product.brand
  comment: "Product brand"

- name: Product Segment
  expr: product.product_segment
  comment: "Product segment"

# --- Transaction attributes (from fact_sales) ---
- name: Payment Mode
  expr: source.payment_mode
  comment: "Payment method used for the sale"

- name: Sales Channel
  expr: source.sales_channel
  comment: "Channel the sale came through"

- name: Store ID
  expr: source.store_id
  comment: "Store where the sale occurred"

measures:
- name: Transaction Count
  expr: COUNT(1)
  comment: "Number of sales transactions"

- name: Total Units Sold
  expr: SUM(source.quantity)
  comment: "Total units sold"

- name: Total Sales
  expr: SUM(source.selling_price)
  comment: "Total sales amount before discounts (gross sales)"

- name: Total Discounts
  expr: SUM(source.discount_amount)
  comment: "Total discount amount given"

- name: Revenue
  expr: SUM(source.net_amount)
  comment: "Revenue: total net sales after discounts"

- name: Average Order Value
  expr: AVG(source.net_amount)
  comment: "Average net revenue per transaction (AOV)"

- name: Unique Customers
  expr: COUNT(DISTINCT source.customer_id)
  comment: "Number of distinct customers with sales"

- name: Discount Rate
  expr: SUM(source.discount_amount) / NULLIF(SUM(source.selling_price), 0)
  comment: "Total discounts as a share of total sales"

- name: Units per Transaction
  expr: SUM(source.quantity) / NULLIF(COUNT(1), 0)
  comment: "Average units sold per transaction"

- name: Revenue per Customer
  expr: SUM(source.net_amount) / NULLIF(COUNT(DISTINCT source.customer_id), 0)
  comment: "Average revenue per unique customer"

  
$$